# Aula 1: Arquitetura RAG na Prática com Gemini

## O que vamos aprender


*   Como o RAG resolve problemas reais dos LLMs
*   Diferenças práticas entre RAG e prompting tradicional
*   Primeiros passos com LangChian e Google Gemini
*   Como Carregar e processar documentos PDF



## Por que RAG é essencial no mercado?



1.   **Dados sempre atualizados** - Sem retreinar modelos
2.   **Respostas verificáveis** - Com fontes de informação
3.   **Custo otimizado** - Evita Fine-tuning caro
4.   **Aplicações reais**  Chatbots de suporte, análise de documentos, assistentes internos



# New Section

## 0. Configuração do Ambiente

Primeiro, vamos instalar as bibliotecas necessárias. Usaremos:
- `langchain` e `langchain-google-genai` para interagir com Gemini
- `pypdf` para ler o conteúdo do nosso arquivo PDF.

In [ ]:
!pip install -q langchain langchain-google-genai==2.1.6 pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [4]:
!pip install langchain-google-genai==2.1.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.2/457.2 kB 18.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.2.0 requires langchain-core<2.0.0,>=1.2.1, but you have langchain-core 0.3.81 which is incompatible.
langgraph-prebuilt 1.0.5 requires langchain-core>

- Onde obter a chave google: https://aistudio.google.com/apikey

In [2]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

## 1. Prompting Tradicional vs RAG - Comparação Prática

Vamos ver na prática a diferença entre usar apenas um LLM (prompting tradicional) e usar RAG.

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

ModuleNotFoundError: No module named 'langchain_google_genai'

In [ ]:
llm

ChatGoogleGenerativeAI(model='models/gemini-2.5-flash-lite', google_api_key=SecretStr('**********'), temperature=0.0, client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x795c7c358530>, default_metadata=(), model_kwargs={})

In [ ]:
#Exemplo 1: Prompting tradicional (sem RAG)

pergunta = "Qual é a politica de home office da nossa empresa"

prompt_tradicional = ChatPromptTemplate.from_template(
    "Responda a seguinte pergunta: {pergunta}"
)

In [ ]:
chain_tradicional = prompt_tradicional | llm

resposta_tradicional = chain_tradicional.invoke({"pergunta": pergunta})

In [ ]:
print(resposta_tradicional.content)

Para responder a essa pergunta de forma precisa, preciso de mais informações sobre a sua empresa. **Não tenho acesso a informações internas de empresas.**

No entanto, posso te dar um guia sobre como você pode descobrir qual é a política de home office da sua empresa e quais são os pontos que você deve procurar.

**Para descobrir a política de home office da sua empresa, você pode:**

1.  **Consultar o Departamento de Recursos Humanos (RH):** Esta é a fonte mais confiável. O RH é responsável por gerenciar as políticas de pessoal da empresa e terá a informação exata.
2.  **Verificar o Manual do Colaborador ou Código de Conduta:** Muitas empresas incluem suas políticas de trabalho remoto nesses documentos. Procure por seções sobre "Trabalho Remoto", "Home Office", "Flexibilidade de Trabalho" ou algo similar.
3.  **Perguntar ao seu Gestor Direto:** Seu líder imediato provavelmente saberá ou poderá te direcionar para a pessoa certa que possui essa informação.
4.  **Consultar a Intranet ou 

# Aula 2: Armazenamento Veotrial com FAISS, Chroma e Pinecone

## O que vamos aprender
* Como aramazenas e buscar embeddings eficientemente com FAISS e Chroma (locais) e Pinecone (nuvem)
* A importância dos metadados para filtros precisos.
* As diferenças práticas entre Índices Flat e HNSW.
* Como migrar de uma solução local para uma solução gerenciada na nuvem.

## Por que armazenamento vetorial é crucial?
* **Escala:** Buscas em milhões de documentos em milissegundos.
* **Persistência:** Não recalcular embeddings a cada execução.
* **Filtros:** Combinar busca semânticas com regras de negócio.
* **Produção:** Soluções em nuvem como Pinecone eoferecem escalabilidade e gerenciamento simplificado.

## 0. Configuração do Ambiente

Vamos instalar as bliblitecas e configurar nossa chave de API do gemini.

**Importante:** Para a seção do pinecone. você precisa de uma conta e uma chave de API.

1. Crie uma conta em [pinceone.io](https://www.pinecone.io/)
2. Crie um índicie (por exemplo, `langchain-rag`) com a dimensão correta para o modelo de embeddinng que estamos usando (Gemini 768).
3. Na seção "API Keys", crie uma chave.

Adicione suas chaves ao arquivo `.env`:

`GOOGLE_API_KEY="sua-chave-do-google-aqui"`
`PINECONE_API_KEY="sua-chave-do-pinecone-aqui"`

In [ ]:
!pip install langchain langchain-google-genai==2.1.6 faiss-cpu chromadb langchain-pinecone pinecone-client

  Using cached langchain_google_genai-2.1.6-py3-none-any.whl.metadata (7.0 kB)
  Using cached faiss_cpu-1.12.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447

In [ ]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.schema import Document

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")


## FAISS

In [ ]:
embeddings.embed_query("Política de home office da empresa")

[0.06417398899793625,
 -0.01762194000184536,
 -0.06427328288555145,
 0.02658555656671524,
 0.060529351234436035,
 0.030340494588017464,
 0.0029144815634936094,
 -0.0354776494204998,
 -0.012707725167274475,
 0.05717604607343674,
 -0.01755794882774353,
 0.006505637429654598,
 -0.03459867835044861,
 0.005386482924222946,
 0.0018029307248070836,
 -0.02684132754802704,
 0.022470351308584213,
 0.014389364048838615,
 0.0030616712756454945,
 -0.04295593500137329,
 0.03674696013331413,
 0.0097909951582551,
 -0.028526276350021362,
 0.010601318441331387,
 0.02377132698893547,
 -0.016931399703025818,
 0.012199308723211288,
 -0.055948205292224884,
 -0.033063553273677826,
 0.0170906912535429,
 -0.03612414747476578,
 0.03503110259771347,
 -0.06779661029577255,
 0.03357503190636635,
 -0.011242709122598171,
 -0.012224187143146992,
 -0.02246846817433834,
 -0.006236195098608732,
 -0.004482307471334934,
 0.0063132173381745815,
 0.0396149680018425,
 -0.015136351808905602,
 -0.05227455869317055,
 0.02534984

In [ ]:
documentos_empresa = [
    Document(
        page_content="Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.",
        metadata={"tipo": "política", "departamento": "RH", "ano": 2024, "id_doc": "doc001"}
    ),
    Document(
        page_content="Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.",
        metadata={"tipo": "processo", "departamento": "Financeiro", "ano": 2023, "id_doc": "doc002"}
    ),
    Document(
        page_content="Guia de TI: Para configurar a VPN, acesse vpn.nossaempresa.com e siga as instruções para seu sistema operacional.",
        metadata={"tipo": "tutorial", "departamento": "TI", "ano": 2024, "id_doc": "doc003"}
    ),
    Document(
        page_content="Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.",
        metadata={"tipo": "política", "departamento": "RH", "ano": 2022, "id_doc": "doc004"}
    )
]

In [ ]:
from langchain_community.vectorstores import FAISS
import faiss

d = 768
index_hnsw = faiss.IndexHNSWFlat(d, 32)

In [ ]:
faiss_db = FAISS.from_documents(documentos_empresa, embeddings)

pergunta = "Como peço minhas férias?"
resultados = faiss_db.similarity_search(pergunta, k=2)



In [ ]:
print(f'\n Pergunta: {pergunta}')
print("\n Documentos mais relevantes (FAISS):")
for doc in resultados:
  print(f'- {doc.page_content}')
  print(f"(Metadados: {doc.metadata})")


 Pergunta: Como peço minhas férias?

 Documentos mais relevantes (FAISS):
- Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
(Metadados: {'tipo': 'política', 'departamento': 'RH', 'ano': 2024, 'id_doc': 'doc001'})
- Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.
(Metadados: {'tipo': 'processo', 'departamento': 'Financeiro', 'ano': 2023, 'id_doc': 'doc002'})


## Chroma DB

In [ ]:
from langchain_community.vectorstores import Chroma

chroma_db = Chroma.from_documents(
    documents=documentos_empresa,
    embedding=embeddings
)


In [ ]:
resultados = chroma_db.similarity_search(pergunta, k=2)

for doc in resultados:
  print(f'- {doc.page_content}')
#  print(f"(Metadados: {doc.metadata})")

- Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
- Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.


In [ ]:
pergunta_rh = "Quais são as regras da empresa?"

resultado_filtrado = chroma_db.similarity_search(
    pergunta_rh,
    k=2,
    filter={"$and": [{"departamento": "RH"}, {"tipo": "política"}]}
)



In [ ]:
print(f'\n  Perguunta: {pergunta_rh} ciom filtro para políticas de RH')
print("\n  Documentos mais relevantes e filtrados (Chroma):")
for doc in resultado_filtrado:
  print(f'- {doc.page_content}')
  print(f"Departamento: {doc.metadata['departamento']}, Tipo: {doc.metadata['tipo']}")


  Perguunta: Quais são as regras da empresa? ciom filtro para políticas de RH

  Documentos mais relevantes e filtrados (Chroma):
- Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.
Departamento: RH, Tipo: política
- Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
Departamento: RH, Tipo: política


## Pinecone


In [ ]:
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

In [ ]:
from langchain_pinecone import Pinecone
from pinecone import Pinecone as PineconeClient
from pinecone import ServerlessSpec

index_name = "langchain-rag"

pinecone_client = PineconeClient(api_key=os.environ["PINECONE_API_KEY"])

spec = ServerlessSpec(cloud='aws', region='us-east-1')

In [ ]:
# Verificando se o índice existe. Se não, ele será criado.
# Nome do seu índice no Pinecone
index_name = "langchain-rag"
if index_name not in pinecone_client.list_indexes().names():
    pinecone_client.create_index(
        name=index_name,
        dimension=d,
        metric="cosine",
        spec=spec
    )
    print(f"Índice '{index_name}' criado no Pinecone.")

    pinecone_db = Pinecone.from_documents(
        documentos_empresa,
        embeddings,
        index_name=index_name
    )

    print(f"Documentos adicionados ao índice '{index_name}'")

else:
    print(f"Conectando ao índice existente '{index_name}'.")
    # Se o índice já existe, apenas carregamos
    pinecone_db = Pinecone.from_existing_index(
        index_name,
        embeddings
    )

Índice 'langchain-rag' criado no Pinecone.
Documentos adicionados ao índice 'langchain-rag'


In [ ]:
if pinecone_db:
  # Busca por similaridade
  pergunta_ti = "Como configuro a VPN?"
  resultados_pinecone = pinecone_db.similarity_search(pergunta_ti, k=2)
  print(f"\n Pergunta: {pergunta_ti}")
  print(f"\n Documentos mais relevantes (Pinecone):")
  for doc in resultados_pinecone:
    print(f"- {doc.page_content}")
    print(f"(Metadados: {doc.metadata})")

  # Busca com filtro (Pinecone também suporta)
  resultados_pinecone_filtrados = pinecone_db.similarity_search(
      "informações sobre regras",
      k=2,
      filter={"tipo": "política"})

  print(f"\n Pergunta: 'Informações sobre regras' com filtro para tipo='politica'")
  print(f"\n Documentos relevantes e filtrados (Pinecone):")
  for doc in resultados_pinecone_filtrados:
    print(f"- {doc.page_content}")
    print(f"(Tipo: {doc.metadata['tipo']})")


 Pergunta: Como configuro a VPN?

 Documentos mais relevantes (Pinecone):
- Guia de TI: Para configurar a VPN, acesse vpn.nossaempresa.com e siga as instruções para seu sistema operacional.
(Metadados: {'ano': 2024.0, 'departamento': 'TI', 'id_doc': 'doc003', 'tipo': 'tutorial'})
- Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.
(Metadados: {'ano': 2023.0, 'departamento': 'Financeiro', 'id_doc': 'doc002', 'tipo': 'processo'})

 Pergunta: 'Informações sobre regras' com filtro para tipo='politica'

 Documentos relevantes e filtrados (Pinecone):
- Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.
(Tipo: política)
- Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
(Tipo: política)


# Aula 3: Embeddings de Alta Performance

## O que vamos aprender:

* Comparar modelos de API (Gemini) vc. modelos locais (Hugging Face) em custo, velocidade e qualidade
* Aumentar a eficiência do processamento em 10x com **Batch Processing**.
* Economizar custos e reduzir a latência com Cahce de **Embeddings**.


## Por que a perfomance dos embeddings é crucial?

* Qualidade da Busca: A precisão do seu RAG depende diretamente da qualidade dos embeddings.
* Custo Operacional: Modelos locais podem reduzir drasticamente os custos de API em larga escala.
* Velocidade (Latência): O tempo para gerar embeddings impacta a velocidade de indexação e a resposta ao usuário.
* Privacidade: Modelos locais garantem que dados sensíveis permaneçãm na sua infraestrutura

## 0. Configuração

Instalamos as bibliotecas e configuramos a chave de API do Google

In [ ]:
!pip install langchain langchain-google-genai sentence-transformers scikit-learn langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-gene

In [ ]:
import os
import time
import numpy as np

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

NameError: name 'userdata' is not defined

## 1. Comparativo de Modelos: Gemini (API) vs. Hugging Face (Local)

Vamos comparar um modelo de ponta via API (Google Gemini) com modelos open-source populares que rodam localmente.

- Goggle Gemini (`embedding-001`): Modelo de alta qualidade, acessado via API
- `all-MiniLM-L6-v2`: um modelo local muito popular, leve e rápudo. Ótimo para tarefas gerais onde a velocidade é importante.
- `BAII/bge-large-en-v1.5`: Um dos melhores modelos open-source no MTEB Leaderboard. Mais pesaod, mas com qualidade semântica superior.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import time

In [ ]:
textos_teste = [
    "Qual é a política de férias da nossa empresa?",
    "Preciso de um relatório de despesas de viagem.",
    "Como configuro o acesso à rede privada virtual (VPN)?",
    "Onde encontro o código de conduta da organização?",
    "Quero entender o processo de avaliação de performance."
]

In [ ]:
## Google Gemini Embeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

start_time = time.time()
embeddings_gemini = gemini_embeddings.embed_documents(textos_teste)
end_time = time.time()

print(f"Tempo de processamento: {end_time - start_time} segundos")
print(f"  - Dimensões do vetor: {len(embeddings_gemini[0])}")


Tempo de processamento: 1.2903366088867188 segundos
  - Dimensões do vetor: 768


In [ ]:
## all-MiniLM-L6-v2

minilm_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

start_time = time.time()
embeddings_minilm = minilm_embeddings.embed_documents(textos_teste)
end_time = time.time()

print(f"Tempo de processamento: {end_time - start_time} segundos")
print(f"  - Dimensões do vetor: {len(embeddings_minilm[0])}")
#


/tmp/ipython-input-896937428.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  minilm_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warn

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Tempo de processamento: 0.19377803802490234 segundos
  - Dimensões do vetor: 384


In [ ]:
## bge-large-en-v1.5

bge_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

start_time = time.time()
embeddings_bge = bge_embeddings.embed_documents(textos_teste)
end_time = time.time()

print(f"Tempo de processamento: {end_time - start_time} segundos")
print(f"  - Dimensões do vetor: {len(embeddings_bge[0])}")
#

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Tempo de processamento: 1.3545808792114258 segundos
  - Dimensões do vetor: 1024


## Análise de Qualidade Semântica

Agora vamos ver qual modelo entende melhor uma pergunta semânticamente similar, mas com palavras diferentes.

In [ ]:
pergunta = "Quero tirar uns dias de folga do trabalho."

embedding_pergunta_gemini = gemini_embeddings.embed_query(pergunta)
embedding_pergunta_minilm = minilm_embeddings.embed_query(pergunta)
embedding_pergunta_bge = bge_embeddings.embed_query(pergunta)

In [ ]:
modelos = {
    "Gemini": (embedding_pergunta_gemini , embeddings_gemini),
    "MiniLM": (embedding_pergunta_minilm, embeddings_minilm),
    "BGE-Large": (embedding_pergunta_bge, embeddings_bge)
}


print(pergunta)

Quero tirar uns dias de folga do trabalho.


In [ ]:
for nome, (emb_q, emb_docs) in modelos.items():
  similaridades = cosine_similarity([emb_q], emb_docs)[0]
  doc_e_similares = sorted(
      zip(textos_teste, similaridades), key=lambda x: x[1], reverse=True
      )
  print(f"---  Ranking para o modelo {nome} ---")
  for i, (doc, sim) in enumerate(doc_e_similares[:3], 1):
    print(f"  {i}. (Score: {sim:.3f}) {doc}")
  print()

---  Ranking para o modelo Gemini ---
  1. (Score: 0.824) Quero entender o processo de avaliação de performance.
  2. (Score: 0.811) Preciso de um relatório de despesas de viagem.
  3. (Score: 0.761) Qual é a política de férias da nossa empresa?

---  Ranking para o modelo MiniLM ---
  1. (Score: 0.496) Quero entender o processo de avaliação de performance.
  2. (Score: 0.469) Onde encontro o código de conduta da organização?
  3. (Score: 0.465) Qual é a política de férias da nossa empresa?

---  Ranking para o modelo BGE-Large ---
  1. (Score: 0.646) Qual é a política de férias da nossa empresa?
  2. (Score: 0.645) Preciso de um relatório de despesas de viagem.
  3. (Score: 0.621) Onde encontro o código de conduta da organização?



## Caching de Embeddings: Economia e Velocidade

In [ ]:
from langchain.storage import LocalFileStore
from langchain.embeddings import CacheBackedEmbeddings

store = LocalFileStore("./cache/")

embedder_principal = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

cache_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embedder_principal,
    store,
    namespace='gemini_cache'
)


/usr/local/lib/python3.12/dist-packages/langchain/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [ ]:
textos_para_cache = ["Olá, mundo!", "Testando o cache de embeddings.", "Olá, mundo!"]
start_time = time.time()
embeddings_result_1 = cache_embeddings.embed_documents(textos_para_cache)
end_time = time.time()
print(f" Tempo de execução: {end_time - start_time:.4f} segundos.")

 Tempo de execução: 0.0044 segundos.


## Batch Processing para Indexação em Larga Escala

In [ ]:
documentos_grandes = [f"Este é o documento de teste número {i}." for i in range(1000)]

bge_embedder = HuggingFaceEmbeddings(
    model_name='BAAI/bge-large-en-v1.5',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

batch_sizes = [1, 32, 64, 128]

In [ ]:
for batch_size in batch_sizes:
  start_time = time.time()
  num_batches = len(documentos_grandes) // batch_size
  tempo_estimado = num_batches * (0.1 * batch_size) + (len(documentos_grandes) % batch_size)
  tempo_real = bge_embedder.client.encode(documentos_grandes, batch_size=batch_size)
  end_time = time.time()

  print(f"  -Batch Size: {batch_size:<4} -> Tempo: {end_time - start_time:.2f}s")


  -Batch Size: 1    -> Tempo: 357.22s
  -Batch Size: 32   -> Tempo: 155.97s
  -Batch Size: 64   -> Tempo: 152.66s
  -Batch Size: 128  -> Tempo: 154.71s


# Aula 4: Pipelines Complexos

## Configuração

Vamos instalar as bibliotecas necessárias. Note que a unstructured pode ter dependências adicionais para processar certos tipos de arquivo.

**Atenção**: A instalação de unstructured pode ser demorada.

In [ ]:
!pip install -q unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.6/167.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 3.5 MB/s eta 0:00:00


In [ ]:
!pip install "unstructured[pdf]" duckdb pandas langchain langchain_community langchain_google_genai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 32.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import duckdb
import pandas as pd
from datetime import datetime
from google.colab import userdata

os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')

## Processando PDFs Complexos

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

loader = UnstructuredPDFLoader("relatorio.pdf", mode="elements")

docs_unstructured = loader.load()

print(f"Total de elementos extraídos: {len(docs_unstructured)}\n")

for doc in docs_unstructured:
  print(f"---- TIPO DE ELEMENTO: {doc.metadata.get('category')}----")
  print(doc.page_content)
  print("\n")

Total de elementos extraídos: 75

---- TIPO DE ELEMENTO: Title----
Processando PDFs Complexos com Unstructured


---- TIPO DE ELEMENTO: NarrativeText----
Exercício gerado automaticamente — contém elementos variados: colunas, tabelas, imagens, código, formulários e 'páginas digitalizadas'.


---- TIPO DE ELEMENTO: Title----
Autor: Gerador Automático (cid:127) Data: 23 September 2025


---- TIPO DE ELEMENTO: Title----
Seção 1 — Texto multi-coluna com notas de rodapé e link


---- TIPO DE ELEMENTO: NarrativeText----
Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed non risus. Suspendisse lectus tortor, dignissim sit amet, adipiscing nec, ultricies sed, dolor. Cras elementum ultrices diam. Maecenas ligula massa, varius a, semper congue, euismod non, mi. Proin porttitor, orci nec nonummy molestie, enim est eleifend mi, non fermentum diam nisl sit amet erat. Duis semper. Duis arcu massa, scelerisque vitae, consequat in, pretium a, enim. Pellentesque congue. [1] Lorem ipsum dolor s

## Adicionando Metadados Estratégicos na Carga

In [ ]:
from langchain.schema.document import Document

docs_com_metadados = []

for doc in docs_unstructured:

  novos_metadados = doc.metadata.copy()

  novos_metadados['source'] = 'relatorio.pdf'
  novos_metadados['ingestion_date'] = datetime.now().strftime('%Y-%m-%d')
  novos_metadados['data_owner'] = 'Departamento de Vendas'

  docs_com_metadados.append(
      Document(page_content=doc.page_content, metadata=novos_metadados)
  )

print(f"Total de documentos com metadados: {len(docs_com_metadados)}\n")
print(docs_com_metadados[-1])

Total de documentos com metadados: 75

page_content='1. Documento de exemplo — Gerador Automático. 2. Recursos sobre Unstructured e OCR.' metadata={'source': 'relatorio.pdf', 'coordinates': {'points': ((76.86614, 252.76290000000006), (76.86614, 262.76290000000006), (481.4661399999999, 262.76290000000006), (481.4661399999999, 252.76290000000006)), 'system': 'PixelSpace', 'layout_width': 595.2756, 'layout_height': 841.8898}, 'filename': 'relatorio.pdf', 'last_modified': '2025-09-23T14:41:14', 'page_number': 8, 'languages': ['por'], 'filetype': 'application/pdf', 'parent_id': 'fe70cfcb9d602e3598869c6ce3464f7b', 'category': 'ListItem', 'element_id': '2f88ff0a74861b8c3da48ac92c02de11', 'ingestion_date': '2025-09-23', 'data_owner': 'Departamento de Vendas'}


## Chuking Inteligente com RecursiveCharacterTextSplitter

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,

)

chunks = text_splitter.split_documents(docs_com_metadados)

print(f"Numero de documentos original: {len(docs_com_metadados)}") #Conferir número original
print(f"Numero de documentos após chunking: {len(chunks)}\n") # Conferir número apos chunking

print(chunks[2]) #verificação para ver o resultado

Numero de documentos original: 75
Numero de documentos após chunking: 78

page_content='Autor: Gerador Automático (cid:127) Data: 23 September 2025' metadata={'source': 'relatorio.pdf', 'coordinates': {'points': ((324.40144, 465.81350000000003), (324.40144, 473.81350000000003), (515.6094400000001, 473.81350000000003), (515.6094400000001, 465.81350000000003)), 'system': 'PixelSpace', 'layout_width': 595.2756, 'layout_height': 841.8898}, 'filename': 'relatorio.pdf', 'last_modified': '2025-09-23T14:41:14', 'page_number': 1, 'languages': ['por'], 'filetype': 'application/pdf', 'category': 'Title', 'element_id': 'afb4fffe04d4c67e9f83dee6c1e54f01', 'ingestion_date': '2025-09-23', 'data_owner': 'Departamento de Vendas'}


## Ingestão de Dados de um Banco SQL com DuckDB

In [ ]:
import duckdb
import pandas as pd

# Conectar ao DuckDB (ele cria o arquivo se não existir)
con = duckdb.connect(database=':memory:', read_only=False)

# Criar uma tabela de produtos
con.execute("""
CREATE TABLE produtos (
    id INTEGER,
    nome VARCHAR,
    categoria VARCHAR,
    preco FLOAT,
    estoque INTEGER,
    descricao VARCHAR
);
""")

produtos_df = pd.DataFrame({
    'id': [101, 102, 103, 104],
    'nome': ['Laptop Gamer Z', 'Mouse Óptico Fast', 'Teclado Mecânico Pro', 'Monitor Curvo 34"'],
    'categoria': ['Eletrônicos', 'Acessórios', 'Acessórios', 'Eletrônicos'],
    'preco': [9500.00, 250.00, 800.00, 3200.00],
    'estoque': [15, 120, 60, 25],
    'descricao': [
        'Laptop de alta performance com placa de vídeo dedicada e 32GB RAM.',
        'Mouse com 16.000 DPI e design ergonômico para longas sessões.',
        'Teclado com switches mecânicos, RGB e layout ABNT2.',
        'Monitor ultrawide com alta taxa de atualização e cores vibrantes.'
    ]
})

con.register('produtos_df', produtos_df)
con.execute('INSERT INTO produtos SELECT * FROM produtos_df')

print("Tabela 'produtos' criada e populada com sucesso!")

#verificar os dados
print(con.execute('SELECT * FROM produtos').fetchdf())

Tabela 'produtos' criada e populada com sucesso!
    id                  nome    categoria   preco  estoque  \
0  101        Laptop Gamer Z  Eletrônicos  9500.0       15   
1  102     Mouse Óptico Fast   Acessórios   250.0      120   
2  103  Teclado Mecânico Pro   Acessórios   800.0       60   
3  104     Monitor Curvo 34"  Eletrônicos  3200.0       25   

                                           descricao  
0  Laptop de alta performance com placa de vídeo ...  
1  Mouse com 16.000 DPI e design ergonômico para ...  
2  Teclado com switches mecânicos, RGB e layout A...  
3  Monitor ultrawide com alta taxa de atualização...  


## Transformando Linhas SQL em `Documentos`

In [ ]:

#Query para selecionar dados
df_produtos = con.execute("SELECT * FROM produtos;").fetchdf()

#Lista para armazenas os documentos
docs_sql = []

fonte_de_dados = "banco_de_dados_produtos_duckdb"


for _, row in df_produtos.iterrows():
    # Criar um texto descritivo a partir da linha
    page_content = f"Produto: {row['nome']}, Categoria: {row['categoria']}, Preço: R${row['preco']:.2f}. Em estoque: {row['estoque']} unidades. Descricao: {row['descricao']}"
    # Criar metadados estratégicos
    metadata = {
        'source': 'tabela_produtos_duckdb',
        'produto_id': row['id'],
        'categoria': row['categoria'],
        'preco': row['preco'],
        'ingestion_date': datetime.now().strftime('%Y-%m-%d')
    }
    docs_sql.append(
        Document(page_content=page_content, metadata=metadata)
    )
#Fechando conexão com o banco
con.close()


print(f"Total de documentos gerados a partir do SQL: {len(docs_sql)}\n")
print("Exemplo de documento gerado a partir de uma linha do banco de dados:")
print(docs_sql[0])

Total de documentos gerados a partir do SQL: 4

Exemplo de documento gerado a partir de uma linha do banco de dados:
page_content='Produto: Laptop Gamer Z, Categoria: Eletrônicos, Preço: R$9500.00. Em estoque: 15 unidades. Descricao: Laptop de alta performance com placa de vídeo dedicada e 32GB RAM.' metadata={'source': 'tabela_produtos_duckdb', 'produto_id': 101, 'categoria': 'Eletrônicos', 'preco': 9500.0, 'ingestion_date': '2025-09-23'}


## Unindo os Pipelines e Enviando para o Vector Store

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

documentos_finais = chunks + docs_sql

print(f"Total de documentos a serem indexados: {len(documentos_finais)}")

documentos_filtrados = filter_complex_metadata(documentos_finais)

print(f"Total de documentos filtrados: {len(documentos_filtrados)}")

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

vector_store = Chroma.from_documents(
    documents=documentos_filtrados,
    embedding=embeddings
)


Total de documentos a serem indexados: 82
Total de documentos filtrados: 82


## Testando o resultado final

In [ ]:
pergunta_pdf = "Qual foi a receita com laptops"

resultados_pdf = vector_store.similarity_search(pergunta_pdf, k=2)

print(f"\n Pergunta: {pergunta_pdf}")

for doc in resultados_pdf:
  print(f"- {doc.page_content}")
  print("\n")

# Aula 5: Cadeias de Conversação Robusta

In [ ]:
!pip install langchain langchain-google-genai langchain-community chromadb tiktoken google-generativeai

In [ ]:
# [] Importações necessárias
import os
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
import google.generativeai as genai
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [ ]:
# Criação de documentos de exemplo sobre IA e Machine Learning
documentos_exemplo = [
    """Inteligência Artificial (IA) é um campo da ciência da computação que se concentra
na criação de sistemas capazes de realizar tarefas que normalmente requerem inteligência humana.
Isso inclui aprendizado, raciocínio, percepção e tomada de decisões.""",

    """Machine Learning é uma subárea da IA que permite que computadores aprendam e melhorem
automaticamente através da experiência, sem serem explicitamente programados.
Os algoritmos de ML identificam padrões em dados e fazem previsões.""",

    """Deep Learning é uma técnica de machine learning baseada em redes neurais artificiais
com múltiplas camadas. É especialmente eficaz para tarefas como reconhecimento de imagem,
processamento de linguagem natural e reconhecimento de voz.""",

    """RAG (Retrieval-Augmented Generation) é uma técnica que combina recuperação de informações
com geração de texto. Permite que modelos de linguagem acessem conhecimento externo
para gerar respostas mais precisas e atualizadas.""",

    """LangChain é um framework para desenvolvimento de aplicações com modelos de linguagem.
Facilita a criação de cadeias complexas, gerenciamento de memória e integração
com diferentes fontes de dados.""",

    """Google Gemini é um modelo de linguagem multimodal desenvolvido pelo Google,
capaz de processar texto, imagens e código. Oferece capacidades avançadas de
raciocínio e compreensão contextual."""
]


#conversão para objetos Document
docs = [Document(page_content=doc) for doc in documentos_exemplo]

print(f"Criados {len(docs)} documentos de exemplo")

Criados 6 documentos de exemplo


## Criação do Vector Store com Gemini Embeddings

In [ ]:
#inicialização dos embeddings do google gemini

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=os.environ["GOOGLE_API_KEY"]
    )

# criação do vector store

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db_gemini"
)


print(f" Número de documentos indexados: {vectorstore._collection.count()}")

 Número de documentos indexados: 6


## Gerenciamento de memória

In [ ]:
memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

print("Memoria configurada")
print(memory.k)

Memoria configurada
5


## ConversationalRetrievalChain com Google Gemini

In [ ]:
## Inicialização do modelo google gemini

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature = 0.7,
    convert_system_message_to_human=True
)



## Criação da ConversationalRetrievalChain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k":3}),
    memory=memory,
    return_source_documents=True,
    verbose=True
)

## Testando a conversação

In [ ]:
def fazer_pergunta(pergunta):
    """Função auxiliar para fazer perguntas à cadeia conversacional"""
    print(f"\n Pergunta: {pergunta}")
    print("-" * 50)

    try:
        resultado = qa_chain({"question": pergunta})

        print(f"✅ Resposta: {resultado['answer']}")
        print(f"\n�� Documentos utilizados: {len(resultado['source_documents'])}")

        return resultado
    except Exception as e:
        print(f"❌ Erro: {str(e)}")
        return None

resultado1 = fazer_pergunta("O que é Inteligência artificial")


 Pergunta: O que é Inteligência artificial
--------------------------------------------------


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
Inteligência Artificial (IA) é um campo da ciência da computação que se concentra
na criação de sistemas capazes de realizar tarefas que normalmente requerem inteligência humana.
Isso inclui aprendizado, raciocínio, percepção e tomada de decisões.

Machine Learning é uma subárea da IA que permite que computadores aprendam e melhorem
automaticamente através da experiência, sem serem explicitamente programados.
Os algoritmos de ML identificam padrões em dados e fazem previsões.

Deep Learning é uma técnica de machine learning baseada em redes neurais artificiais
com múltiplas camadas. É especialmente eficaz pa

In [ ]:
resultado2 = fazer_pergunta("Como ela se relaciona com machine learning")


 Pergunta: Como ela se relaciona com machine learning
--------------------------------------------------


> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: O que é Inteligência artificial
Assistant: Inteligência Artificial (IA) é um campo da ciência da computação que se concentra na criação de sistemas capazes de realizar tarefas que normalmente requerem inteligência humana, como aprendizado, raciocínio, percepção e tomada de decisões.
Follow Up Input: Como ela se relaciona com machine learning
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
--------

In [ ]:
resultado3 = fazer_pergunta("E o que Google Gemini como você mencionou")


 Pergunta: E o que Google Gemini como você mencionou
--------------------------------------------------


> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: O que é Inteligência artificial
Assistant: Inteligência Artificial (IA) é um campo da ciência da computação que se concentra na criação de sistemas capazes de realizar tarefas que normalmente requerem inteligência humana, como aprendizado, raciocínio, percepção e tomada de decisões.
Human: Como ela se relaciona com machine learning
Assistant: A Inteligência Artificial (IA) é um campo mais amplo da ciência da computação que visa criar sistemas capazes de realizar tarefas que normalmente exigiriam inteligência humana. O Machine Learning (ML) é uma subárea da IA que permite que computadores aprendam e melhorem com a experiência, sem serem explicitamente program

## Guardrails de Segurança

In [ ]:
import re

class GuardrailsSeguranca:
    def __init__(self):
        self.palavras_proibidas = {
            'senha', 'password', 'cpf', 'rg', 'cartão de crédito',
            'dados pessoais', 'informação confidencial', 'api key',
            'chave de api', 'token de acesso'
        }
        self.padroes_pii = {
            r'\d{3}\.\d{3}\.\d{3}-\d{2}',                                     # Padrão CPF (999.999.999-99)
            r'\d{4}\s?\d{4}\s?\d{4}\s?\d{4}',                                  # Padrão cartão de crédito (16 digitos)
            r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',            # Padrão e-mail
            r'AIza[0-9A-Za-z-_]{35}'                                          # Padrão Google API Key
        }

    def verificar_pergunta(self, pergunta):
      """Verifica se a pergunta contém conteúdo inadequado"""
      pergunta_lower = pergunta.lower()

      # Verificar palavras proibidas
      for palavra in self.palavras_proibidas:
          if palavra in pergunta_lower:
              return False, f"Pergunta contém termo inadequado: {palavra}"

      # Verificar padrões PII
      for padrao in self.padroes_pii:
          if re.search(padrao, pergunta):
              return False, "Pergunta contém informações pessoais"

      return True, "Pergunta aprovada"

    def verificar_resposta(self, resposta):
        """Verifica se a resposta é adequada"""
        resposta_lower = resposta.lower()

        # Verificar se a resposta está no escopo
        termos_escopo = ['ia', 'inteligência artificial', 'machine learning', 'deep learning', 'rag', 'langchain', 'gemini', 'google'] # Tópicos permitidos

        tem_termo_escopo = any(termo in resposta_lower for termo in termos_escopo) # Confere se la tem pelo menos um termo r

        if not tem_termo_escopo and len(resposta) > 50: # Se não for do escopo e ainda for longa
            return False, "Resposta fora do escopo da aplicação" # Bloqueia

        # Verificar se não contém informações sensíveis
        for padrao in self.padroes_pii: # Percorre cada regex de PII
            if re.search(padrao, resposta): # Se encontrar dado sensível
                return False, "Resposta contém informações sensíveis" # Bloqueia

        return True, "Resposta aprovada" # Retorna sucesso se tudo estiver OK


# Inicialização dos guardrails
guardrails = GuardrailsSeguranca() # Cria a instância dos guardrails
print("✅ Guardrails de segurança configurados!")

✅ Guardrails de segurança configurados!


In [ ]:
def pergunta_segura(pergunta):
    """Função que aplica guardrails antes de processar a pergunta"""
    aprovada, mensagem = guardrails.verificar_pergunta(pergunta)

    if not aprovada:
        print(f"�� Pergunta rejeitada: {mensagem}")
        return None

    try:
        # Processar pergunta
        resultado = qa_chain({"question": pergunta})

        # Verificar resposta
        aprovada_resp, mensagem_resp = guardrails.verificar_resposta(
            resultado['answer']
        )

        if not aprovada_resp:
            print(f"�� Resposta rejeitada: {mensagem_resp}")
            return None

        print(f"✅ {mensagem}")
        print(f"✅ {mensagem_resp}")
        print(f"\n Resposta: {resultado['answer']}")

        return resultado
    except Exception as e:
        print(f"❌ Erro ao processar pergunta: {str(e)}")
        return None

# Teste com pergunta adequada
print("\n=== Teste com pergunta adequada ===")
pergunta_segura("Explique sobre Deep Learning")

# Teste com pergunta inadequada
print("\n=== Teste com pergunta inadequada ===")
pergunta_segura("Qual é a sua chave de API?")


=== Teste com pergunta adequada ===


> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: O que é Inteligência artificial
Assistant: Inteligência Artificial (IA) é um campo da ciência da computação que se concentra na criação de sistemas capazes de realizar tarefas que normalmente requerem inteligência humana, como aprendizado, raciocínio, percepção e tomada de decisões.
Human: Como ela se relaciona com machine learning
Assistant: A Inteligência Artificial (IA) é um campo mais amplo da ciência da computação que visa criar sistemas capazes de realizar tarefas que normalmente exigiriam inteligência humana. O Machine Learning (ML) é uma subárea da IA que permite que computadores aprendam e melhorem com a experiência, sem serem explicitamente programados. Em essência, o ML é uma forma de alcançar a IA, permitindo que

## Re-ranking

# Aula 6: Avaliação com LangSmith & RAGAS

In [ ]:
!pip install langchain langchain_google_genai langchain_community chromadb tiktoken google_generativeai ragas langsmith datasets

In [ ]:
# Importações necessárias
import os
import pandas as pd
import numpy as np
from datetime import datetime
import json
import time
import warnings
warnings.filterwarnings('ignore')

# LangChain imports
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.schema import Document
from langchain.text_splitter import CharacterTextSplitter

# RAGAS imports
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

from datasets import Dataset

In [ ]:
import google.generativeai as genai

from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')


In [ ]:
# Criação de documentos de exemplo sobre IA e tecnologia
documentos_conhecimento = [
    """Inteligência Artificial (IA) é um campo da ciência da computação que se concentra
na criação de sistemas capazes de realizar tarefas que normalmente requerem inteligência humana.
Isso inclui aprendizado, raciocínio, percepção e tomada de decisões. A IA pode ser classificada
em IA fraca (específica para tarefas) e IA forte (inteligência geral).""",
    """Machine Learning é uma subárea da IA que permite que computadores aprendam e melhorem
automaticamente através da experiência, sem serem explicitamente programados.
Os algoritmos de ML identificam padrões em dados e fazem previsões. Existem três tipos principais:
aprendizado supervisionado, não supervisionado e por reforço.""",
    """Deep Learning é uma técnica de machine learning baseada em redes neurais artificiais
com múltiplas camadas. É especialmente eficaz para tarefas como reconhecimento de imagem,
processamento de linguagem natural e reconhecimento de voz. As redes neurais profundas
podem ter centenas de camadas e milhões de parâmetros.""",
    """RAG (Retrieval-Augmented Generation) é uma técnica que combina recuperação de informações
com geração de texto. Permite que modelos de linguagem acessem conhecimento externo
para gerar respostas mais precisas e atualizadas. O processo envolve buscar documentos
relevantes e usar essas informações para gerar a resposta final.""",
    """Google Gemini é um modelo de linguagem multimodal desenvolvido pelo Google,
capaz de processar texto, imagens e código. Oferece capacidades avançadas de
raciocínio e compreensão contextual. O Gemini vem em diferentes versões:
Nano, Pro e Ultra, cada uma otimizada para diferentes casos de uso.""",
    """LangChain é um framework para desenvolvimento de aplicações com modelos de linguagem.
Facilita a criação de cadeias complexas, gerenciamento de memória e integração
com diferentes fontes de dados. Oferece componentes modulares para construir
aplicações robustas de IA conversacional."""
]

# Conversão para objetos Document
docs = [Document(page_content=doc) for doc in documentos_conhecimento]
print(f"✅ Criados {len(docs)} documentos de conhecimento")

✅ Criados 6 documentos de conhecimento


In [ ]:
# Criação de dataset de teste para avaliação RAGAS
dados_teste = {
    'question': [
        "O que é Inteligência Artificial?",
        "Como funciona o Machine Learning?",
        "Quais são as aplicações do Deep Learning?",
        "O que é RAG e como funciona?",
        "Quais são as características do Google Gemini?"
    ],
    'ground_truth': [
        "Inteligência Artificial é um campo da ciência da computação focado na criação de sistemas que realizam tarefas que requerem inteligência humana, como aprendizado, raciocínio e percepção.",
        "Machine Learning permite que computadores aprendam automaticamente através da experiência, identificando padrões em dados para fazer previsões.",
        "Deep Learning é eficaz para reconhecimento de imagem, processamento de linguagem natural e reconhecimento de voz, usando redes neurais profundas.",
        "RAG combina recuperação de informações com geração de texto, permitindo que modelos acessem conhecimento externo para respostas mais precisas.",
        "Google Gemini é um modelo multimodal que processa texto, imagens e código, oferecendo capacidades avançadas de raciocínio e vem em versões como Nano, Pro e Ultra."
    ]
}
print("✅ Dataset de teste criado!")
print(f"✅ {len(dados_teste['question'])} perguntas de teste preparadas")

✅ Dataset de teste criado!
✅ 5 perguntas de teste preparadas


## Criação do sistema RAG

In [ ]:
#Inicialização do embeddings do google gemini
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=os.environ["GOOGLE_API_KEY"]
    )

#Criação do vector store
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db_avaliacao"
)

#Criação do modelo Google Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature = 0.3,
    convert_system_message_to_human=True
)

#Configuracao da memoria
memory = ConversationBufferWindowMemory(
    k=3,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

#Criação da cadeia RAG
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k":3}),
    memory=memory,
    return_source_documents=True,
    verbose=False
)

In [ ]:
def executar_rag_e_coletar_dados(perguntas):
    """Executa o sistema RAG e coleta dados para avaliação"""

    resultados = {
        'question': [],
        'answer': [],
        'contexts': [],
        'ground_truth': []
    }

    for i, pergunta in enumerate(perguntas):
        print(f"Processando pergunta {i+1}/{len(perguntas)}: {pergunta}")

        try:
            # Executar RAG
            resultado = rag_chain({"question": pergunta})

            # Extrair contextos dos documentos fonte
            contextos = [doc.page_content for doc in resultado['source_documents']]

            # Armazenar resultados
            resultados['question'].append(pergunta)
            resultados['answer'].append(resultado['answer'])
            resultados['contexts'].append(contextos)
            resultados['ground_truth'].append(dados_teste['ground_truth'][i])

            print(f"Resposta gerada: {resultado['answer'][:100]}...")

        except Exception as e:
            print(f"Erro ao processar pergunta: {str(e)}")
            continue

    return resultados




In [ ]:
# Executar coleta de dados
print("Iniciando coleta de dados para avaliação...")
dados_avaliacao = executar_rag_e_coletar_dados(dados_teste['question'])
#print(f"\nColeta concluída! {len(dados_avaliacao['question'])} exemplos coletados")

Iniciando coleta de dados para avaliação...
Processando pergunta 1/5: O que é Inteligência Artificial?
Resposta gerada: Inteligência Artificial (IA) é um campo da ciência da computação focado na criação de sistemas que p...
Processando pergunta 2/5: Como funciona o Machine Learning?
Resposta gerada: O Machine Learning (ML) é uma subárea da Inteligência Artificial (IA) que permite que computadores a...
Processando pergunta 3/5: Quais são as aplicações do Deep Learning?
Resposta gerada: As aplicações do Deep Learning incluem:

*   Reconhecimento de imagem
*   Processamento de linguagem...
Processando pergunta 4/5: O que é RAG e como funciona?
Resposta gerada: RAG (Retrieval-Augmented Generation) é uma técnica que combina recuperação de informações com geraçã...
Processando pergunta 5/5: Quais são as características do Google Gemini?
Resposta gerada: O Google Gemini é um modelo de linguagem multimodal desenvolvido pelo Google. Suas características i...


In [ ]:
#Preparar dataset para RAGAS
dataset_ragas = Dataset.from_dict(dados_avaliacao)

# Configurar métricas RAGAS
metricas_ragas = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

print("\n Métricas RAGAS configuradas:")
for metrica in metricas_ragas:
    print(f" - {metrica.name}")


 Métricas RAGAS configuradas:
 - faithfulness
 - answer_relevancy
 - context_precision
 - context_recall


In [ ]:
# Executar avaliação RAGAS
print("Iniciando avaliação com RAGAS...")
print("Isso pode levar alguns minutos...")

# Configurar LLM para RAGAS (usando Gemini)
resultado_ragas = evaluate(
    dataset_ragas,
    metrics=metricas_ragas,
    llm=llm,
    embeddings=embeddings
)

print("\n Avaliação RAGAS concluída!")

# Exibir resultados
print("\n === RESULTADOS DA AVALIAÇÃO RAGAS ===")

# Access individual metrics from the EvaluationResult object
print(f"faithfulness: {resultado_ragas['faithfulness'][0]:.4f}")
print(f"answer_relevancy: {resultado_ragas['answer_relevancy'][0]:.4f}")
print(f"context_precision: {resultado_ragas['context_precision'][0]:.4f}")
print(f"context_recall: {resultado_ragas['context_recall'][0]:.4f}")

#Simulação de resultados RAGAS para demonstração
resultado_ragas_simulado = {
    'faithfulness': 0.85,
    'answer_relevancy': 0.78,
    'context_precision': 0.82,
    'context_recall': 0.75
}

print("\n === RESULTADOS SIMULADOS DA AVALIAÇÃO RAGAS === ")
for metrica, valor in resultado_ragas_simulado.items():
  print(f"{metrica}: {valor:.4f}")


Iniciando avaliação com RAGAS...
Isso pode levar alguns minutos...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[9]: IndexError(list index out of range)
ERROR:ragas.executor:Exception raised in Job[1]: IndexError(list index out of range)
ERROR:ragas.executor:Exception raised in Job[5]: IndexError(list index out of range)
ERROR:ragas.executor:Exception raised in Job[13]: IndexError(list index out of range)
ERROR:ragas.executor:Exception raised in Job[17]: IndexError(list index out of range)



 Avaliação RAGAS concluída!

 === RESULTADOS DA AVALIAÇÃO RAGAS ===
faithfulness: 1.0000
answer_relevancy: nan
context_precision: 1.0000
context_recall: 1.0000

 === RESULTADOS SIMULADOS DA AVALIAÇÃO RAGAS === 
faithfulness: 0.8500
answer_relevancy: 0.7800
context_precision: 0.8200
context_recall: 0.7500


In [ ]:
def analisar_metricas_ragas(resultados):
    """
    Analisa e interpreta os resultados de uma avaliação feita pelo RAGAS.

    RAGAS = biblioteca usada para avaliar sistemas RAG (Retrieval-Augmented Generation),
    verificando se as respostas da IA são fiéis, relevantes e se usam bem o contexto.

    Esse código padroniza os resultados (que podem vir em formatos diferentes
    dependendo da versão da lib) e exibe métricas numéricas + uma interpretação qualitativa.
    """

    # ==========================================================
    # 1) Converter o objeto EvaluationResult em dicionário
    # ==========================================================
    # RAGAS mudou algumas vezes a forma como retorna resultados.
    # Este bloco garante que "resultados" sempre será um dict simples.
    if hasattr(resultados, "to_dict"):            # Versões mais novas (>= 0.0.12)
        resultados = resultados.to_dict()
    elif hasattr(resultados, "_scores_dict"):     # Versões intermediárias (0.0.10 - 0.0.11)
        resultados = resultados._scores_dict
    elif not isinstance(resultados, dict):        # Se não for dict ainda, mas for convertível
        resultados = dict(resultados)             # Força a conversão para dict

    # ==========================================================
    # 2) Função auxiliar para pegar valores de métricas
    # ==========================================================
    def pegar_score(chave: str, default: float = 0.0) -> float:
        """
        Busca um score no dicionário de resultados.

        Alguns back-ends do RAGAS retornam:
        - um float direto (ex: 0.85)
        - ou uma lista/ScoreList (ex: [0.85]) → nesse caso pegamos só o primeiro valor.
        """
        valor = resultados.get(chave, default)
        if isinstance(valor, (list, tuple)):  # Se vier lista/tupla, pega o primeiro item
            return float(valor[0])
        return float(valor)                   # Caso contrário, converte direto para float

    # Cabeçalho bonitinho para indicar início da análise
    print("\n === ANÁLISE DETALHADA DAS MÉTRICAS RAGAS ===\n")

    # ==========================================================
    # 3) Métricas avaliadas pelo RAGAS
    # ==========================================================
    # Nem todas as instalações usam os mesmos nomes de métricas,
    # então incluímos variações (context_relevancy vs context_precision, por exemplo).
    metricas = {
        "faithfulness": "Faithfulness (Factualidade)",      # se a resposta é fiel ao contexto
        "answer_relevancy": "Answer Relevancy (Relevância da Resposta)",
        "context_relevancy": "Context Relevancy (Relevância do Contexto)",
        "context_precision": "Context Precision (Precisão do Contexto)",
        "context_recall": "Context Recall (Recall do Contexto)"
    }

    scores_validos = []  # Lista para armazenar os scores encontrados

    # Loop sobre cada métrica
    for chave, nome_legivel in metricas.items():
        if chave not in resultados:
            continue  # Se a métrica não estiver disponível, pula

        score = pegar_score(chave)   # Extrai o valor numérico da métrica
        scores_validos.append(score)

        # Exibe valor numérico formatado
        print(f"**{nome_legivel}: {score:.4f}**")

        # Interpretação qualitativa para ajudar a leitura
        if score >= 0.80:
            print(" Excelente!")
        elif score >= 0.60:
            print(" Moderado.")
        else:
            print(" Baixo.")

    # ==========================================================
    # 4) Score geral (média simples das métricas disponíveis)
    # ==========================================================
    if scores_validos:
        score_geral = float(np.mean(scores_validos))  # média dos scores
        print(f"\n**Score Geral: {score_geral:.4f}**")

        # Interpretação qualitativa do score geral
        if score_geral >= 0.80:
            print(" Sistema RAG com performance excelente!")
        elif score_geral >= 0.60:
            print(" Sistema RAG com performance boa, mas com espaço para melhorias")
        else:
            print(" Sistema RAG precisa de otimizações significativas")
    else:
        print("Nenhuma métrica disponível para análise.")


In [ ]:
analisar_metricas_ragas(resultado_ragas)


 === ANÁLISE DETALHADA DAS MÉTRICAS RAGAS ===

**Faithfulness (Factualidade): 1.0000**
 Excelente!
**Answer Relevancy (Relevância da Resposta): nan**
 Baixo.
**Context Precision (Precisão do Contexto): 1.0000**
 Excelente!
**Context Recall (Recall do Contexto): 1.0000**
 Excelente!

**Score Geral: nan**
 Sistema RAG precisa de otimizações significativas
